# Clinical Dataset Preparation Pipeline

**Task 3 - preparing noisy clinical text for future instruction tuning**

---

## Introduction

Before a language model can be fine-tuned on clinical text, that text has to be
*cleaned*. Real clinical data arrives messy: it is scanned, copied out of web pages,
exported from several systems and duplicated along the way. Training a model on that
mess simply teaches it the mess.

This notebook builds a small, simple **preprocessing pipeline**. It starts from a real
public dataset of medical records, deliberately adds noise, then cleans it all back out
and exports the result in the format used for instruction tuning.

> **This project uses a publicly available clinical/medical text dataset. Noise is
> artificially introduced for demonstrating the preprocessing pipeline.**

### The plan

```
REAL MEDICAL DATASET (MTSamples)
        |
        v   select ~250 records
        |
        v   ADD ARTIFICIAL NOISE
        |     HTML tags, OCR artifacts, duplicates,
        |     empty records, short records, long records
        |
        v   statistics BEFORE cleaning
        |
        v   clean HTML  ->  fix OCR  ->  normalize whitespace
        v   remove empty -> remove short -> remove long -> remove duplicates
        |
        v   statistics AFTER cleaning
        |
        v   convert to instruction format
        |
   clinical_instruction_dataset.jsonl  +  cleaned_clinical_dataset.csv
```

### Why add noise on purpose?

Because it makes the cleaning **checkable**. We know exactly which records we damaged, so
we can prove the pipeline fixed them. If we simply downloaded a messy dataset we would
have no way of knowing whether the cleaning had actually worked.

### Tools

Only `pandas`, `re` (regular expressions) and `json`. No machine learning, no NLP
libraries, no APIs.

---
## Import Libraries

| Library | What we use it for |
|---|---|
| `pandas` | holding the dataset in a table (a "DataFrame") |
| `re` | regular expressions - pattern-based find and replace |
| `json` | writing the final dataset in JSON Lines format |
| `os` | checking whether the offline backup file exists |

In [1]:
import pandas as pd   # tables (DataFrames)
import re             # regular expressions, for finding and replacing patterns
import json           # for writing the final dataset as JSON
import os             # for checking whether a file exists

# Show more of the text in each column instead of cutting it off with "..."
pd.set_option("display.max_colwidth", 90)

print("Libraries imported successfully.")
print("pandas version:", pd.__version__)

Libraries imported successfully.
pandas version: 2.2.3


---
## STEP 1 - Obtain a Clinical Dataset

### The dataset: MTSamples

**MTSamples** is a public collection of ~5,000 sample medical transcription reports -
the kind of documents used to train medical transcriptionists. Each row describes a real
clinical encounter or procedure across specialties such as surgery, cardiology,
radiology, orthopaedics and neurology.

| | |
|---|---|
| **Name** | MTSamples (Medical Transcription Samples) |
| **Source** | `https://raw.githubusercontent.com/socd06/medical-nlp/master/data/mtsamples.csv` |
| **Original site** | mtsamples.com |
| **Size** | 4,999 records, 6 columns |
| **Records used here** | 250 |
| **Content** | clinical descriptions - procedures, symptoms, diagnoses, consultation summaries |
| **Licence / access** | public, no account or API key needed |

### On patient privacy

MTSamples reports are **published sample documents, already de-identified** - they carry
no names, addresses, phone numbers, dates of birth or record numbers. We use only the
`description` column, and STEP 2 runs an automatic check for identifying information so
this is verified rather than assumed.

### Which column, and why

The file has six columns. Two contain clinical text:

- `transcription` - the full report. Median length **2,667** characters, up to 18,425.
- `description` - a one or two sentence summary of the encounter. Median length **120**
  characters, maximum 492.

We use **`description`**. The reason is practical: this pipeline filters out records
longer than 1,000 characters, and almost every full `transcription` exceeds that, so the
filter would delete the entire dataset. The `description` column is real clinical text of
a sensible length for this exercise.

### If the download fails

The notebook tries the live URL first. If there is no internet connection it falls back
to a copy of the same 250 records saved in `data/`, so the notebook always runs.

In [2]:
# Where to get the data from.
DATASET_URL = "https://raw.githubusercontent.com/socd06/medical-nlp/master/data/mtsamples.csv"
BACKUP_FILE = "data/mtsamples_sample_250.csv"   # offline copy of the same 250 records

# try / except means: attempt the first block, and if it raises an error run the second.
# Here it lets the notebook work with or without an internet connection.
try:
    raw_df = pd.read_csv(DATASET_URL)
    print("Downloaded the dataset from the internet.")
    print("Full dataset shape:", raw_df.shape)

    # Keep only the description column and rename it to clinical_text.
    df = raw_df[["description"]].rename(columns={"description": "clinical_text"})

    # Select 250 records. random_state=42 fixes the random choice, so this notebook
    # picks the SAME 250 records every time it is run (this is called reproducibility).
    df = df.sample(250, random_state=42).reset_index(drop=True)

except Exception as error:
    print("Could not download the dataset:", error)
    print("Falling back to the offline copy instead.")
    df = pd.read_csv(BACKUP_FILE)

print()
print("Number of records:", len(df))

Downloaded the dataset from the internet.
Full dataset shape: (4999, 6)

Number of records: 250


---
## STEP 2 - Inspect the Original Dataset

Always look at data before doing anything to it.

`df.info()` reports the number of rows, the column names, how many values are missing,
and the data type of each column.

In [3]:
df.head()

,clinical_text
0,An example/template for meatotomy.
1,"Normal physical exam template. Normal appearance for chronological age, does not app..."
2,Neurologic consultation was requested to assess and assist with seizure medication.
3,"Multiple sharp force injuries, involving chest and abdomen, multiple incised-stab wou..."
4,The patient with atypical type right arm discomfort and neck discomfort.


In [4]:
print("Number of records:", len(df))
print()
df.info()

Number of records: 250

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 1 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clinical_text  250 non-null    object
dtypes: object(1)
memory usage: 2.1+ KB


In [5]:
# Print a few complete records so we can see what the clinical text actually looks like.
print("SAMPLE CLINICAL RECORDS")
print("=" * 78)
for i in range(5):
    print(f"\nRecord {i}:")
    print(" ", df.loc[i, "clinical_text"])

SAMPLE CLINICAL RECORDS

Record 0:
   An example/template for meatotomy.

Record 1:
   Normal physical exam template.  Normal appearance for chronological age, does not appear chronically ill.

Record 2:
   Neurologic consultation was requested to assess and assist with seizure medication.

Record 3:
   Multiple sharp force injuries, involving chest and abdomen, multiple incised-stab wounds of the neck, and multiple incised or cutting wounds.

Record 4:
   The patient with atypical type right arm discomfort and neck discomfort.


### Privacy check

The claim that this data is de-identified should be verified, not assumed. The cell below
searches every record for the patterns that personal information usually takes.

In [6]:
# Regular expressions describing common kinds of identifying information.
pii_patterns = {
    "phone number":   r"\b\d{3}[-.]\d{3}[-.]\d{4}\b",
    "SSN":            r"\b\d{3}-\d{2}-\d{4}\b",
    "email address":  r"\S+@\S+\.\w+",
    "date":           r"\b\d{1,2}/\d{1,2}/\d{2,4}\b",
    "record number":  r"\bMRN\b",
    "personal name":  r"\b(?:Mr|Mrs|Ms|Dr)\.\s+[A-Z][a-z]+",
}

text_column = df["clinical_text"].fillna("").astype(str)

print("PRIVACY CHECK - searching all", len(df), "records")
print("=" * 78)
total_found = 0
for label, pattern in pii_patterns.items():
    matches = text_column.str.contains(pattern, regex=True).sum()
    total_found += matches
    print(f"  {label:16s}: {matches} found")

print()
if total_found == 0:
    print("No identifying information found. Safe to continue.")
else:
    print("WARNING: possible identifying information found - review before continuing.")

PRIVACY CHECK - searching all 250 records
  phone number    : 0 found
  SSN             : 0 found
  email address   : 0 found
  date            : 0 found
  record number   : 0 found
  personal name   : 0 found

No identifying information found. Safe to continue.


---
## STEP 3 - Create a Copy of the Clean Dataset

We keep three DataFrames, and it is worth being clear about why:

| Name | What it holds |
|---|---|
| `df` | the original download - never modified |
| `clean_df` | a copy of the original, kept for comparison at the end |
| `noisy_df` | the working copy that we damage and then clean |

`.copy()` matters here. Writing `noisy_df = clean_df` would create a second *name for the
same table*, so changing one would change the other. `.copy()` makes a genuinely separate
table.

In [7]:
clean_df = df.copy()      # untouched reference copy
noisy_df = clean_df.copy()  # the copy we are about to damage

print("clean_df records:", len(clean_df))
print("noisy_df records:", len(noisy_df))

clean_df records: 250
noisy_df records: 250


In [43]:
# Save the original 250 selected MTSamples records
df.to_csv("raw_clinical_data.csv", index=False, encoding="utf-8")

print("Saved raw_clinical_data.csv")
print("Records:", len(df))

Saved raw_clinical_data.csv
Records: 250


### The real data is already imperfect

Before adding anything, it is worth measuring what is *already* wrong with the real
dataset. This is an important point for understanding the results later: some of the
records the pipeline removes at the end were never noise we added - they came that way.

In [8]:
existing_text = clean_df["clinical_text"].fillna("").astype(str).str.strip()
existing_lengths = existing_text.str.len()

print("PROBLEMS ALREADY PRESENT IN THE REAL DATA")
print("=" * 78)
print("  Empty records            :", (existing_text == "").sum())
print("  Records under 20 chars   :", (existing_lengths < 20).sum())
print("  Records over 1000 chars  :", (existing_lengths > 1000).sum())
print("  Exact duplicate records  :", existing_text.duplicated().sum())
print()
print("  Shortest record:", existing_lengths.min(), "characters")
print("  Longest record :", existing_lengths.max(), "characters")
print()
print("Real datasets are already noisy before anyone adds anything to them.")

PROBLEMS ALREADY PRESENT IN THE REAL DATA
  Empty records            : 0
  Records under 20 chars   : 1
  Records over 1000 chars  : 0
  Exact duplicate records  : 4

  Shortest record: 19 characters
  Longest record : 392 characters

Real datasets are already noisy before anyone adds anything to them.


---
## STEP 4 - Add HTML Tags

When text is copied from a web page or exported from a records system, HTML tags often
come along with it.

> `Patient reports fever and cough.` becomes `<p>Patient reports fever and cough.</p>`

The tags are formatting instructions for a browser, not part of the clinical note, so
they must be removed later.

We modify **existing records** from the real dataset - we do not invent new medical text.

In [9]:
# Wrap the first 15 records in HTML tags, using a few different tags for variety.
html_tags = ["p", "div", "span"]

for position, i in enumerate(noisy_df.index[:15]):
    tag = html_tags[position % len(html_tags)]   # cycle through p, div, span, p, div, ...
    original = noisy_df.loc[i, "clinical_text"]
    noisy_df.loc[i, "clinical_text"] = f"<{tag}>{original}</{tag}>"

html_damaged_count = 15

print("Added HTML tags to", html_damaged_count, "records. First 3:\n")
for i in range(3):
    print(" ", noisy_df.loc[i, "clinical_text"][:100])

Added HTML tags to 15 records. First 3:

  <p> An example/template for meatotomy.</p>
  <div> Normal physical exam template.  Normal appearance for chronological age, does not appear chron
  <span> Neurologic consultation was requested to assess and assist with seizure medication.</span>


---
## STEP 5 - Add OCR Artifacts

**OCR** (Optical Character Recognition) is the software that turns a *scanned image* of a
document into text. Many hospital records are scanned paper forms, so OCR is very common
in clinical data - and it makes predictable mistakes, because some characters look almost
identical on paper.

> **OCR artifacts are simulated by replacing selected characters with visually similar
> numbers.**

| OCR reads | Should be | Why it happens |
|---|---|---|
| `Pat1ent` | `Patient` | digit **1** looks like letter **i** |
| `d1abetes` | `diabetes` | same confusion |
| `hypertens1on` | `hypertension` | same confusion |

We only apply these to records that actually contain the words, and only to the first 20
such records, so the effect stays easy to inspect.

In [10]:
# The mistakes a scanner might make: correct word -> what OCR produced.
ocr_errors = {
    "Patient": "Pat1ent",
    "patient": "pat1ent",
    "diabetes": "d1abetes",
    "hypertension": "hypertens1on",
}


def add_ocr_errors(text):
    """Damage a piece of text by introducing the OCR mistakes listed above."""
    for correct, wrong in ocr_errors.items():
        text = text.replace(correct, wrong)
    return text


# Find the records that contain at least one of the words we can damage.
contains_target_word = noisy_df["clinical_text"].str.contains(
    "|".join(ocr_errors.keys()), regex=True, na=False
)
ocr_target_rows = noisy_df.index[contains_target_word][:20]   # damage the first 20 only

for i in ocr_target_rows:
    noisy_df.loc[i, "clinical_text"] = add_ocr_errors(noisy_df.loc[i, "clinical_text"])

ocr_damaged_count = len(ocr_target_rows)

print("Added OCR artifacts to", ocr_damaged_count, "records. First 3:\n")
for i in ocr_target_rows[:3]:
    print(" ", noisy_df.loc[i, "clinical_text"][:100])

Added OCR artifacts to 20 records. First 3:

  <div> The pat1ent with atypical type right arm discomfort and neck discomfort.</div>
  <div> The pat1ent is a 60-year-old female pat1ent who off and on for the past 10 to 12 months has ha
   The pat1ent is a 58-year-old African-American right-handed female with 16 years of education who wa


---
## STEP 5b - Add Messy Whitespace

One more common problem: stray spaces and line breaks, which appear when text is copied
between systems. We add them to 10 records.

In [11]:
# Insert extra spaces and line breaks into 10 records.
whitespace_target_rows = noisy_df.index[20:30]

for i in whitespace_target_rows:
    original = noisy_df.loc[i, "clinical_text"]
    # Replace the first space with several spaces and a line break, and pad the ends.
    noisy_df.loc[i, "clinical_text"] = "   " + original.replace(" ", "    \n", 1) + "   "

whitespace_damaged_count = len(whitespace_target_rows)

print("Added messy whitespace to", whitespace_damaged_count, "records. First 2:\n")
for i in whitespace_target_rows[:2]:
    # repr() makes spaces and \n line breaks visible instead of invisible
    print(" ", repr(noisy_df.loc[i, "clinical_text"])[:100])

Added messy whitespace to 10 records. First 2:

  '       \nAn 84-year-old woman with a history of hypertens1on, severe tricuspid regurgitation with m
  '       \nLeft-sided large hemicraniectomy for traumatic brain injury and increased intracranial pre


---
## STEP 6 - Add Duplicate Records

> **Some records are intentionally duplicated to demonstrate duplicate removal.**

Duplicates happen when data is merged from several exports. We add two kinds:

1. **10 obvious duplicates** - exact copies of records as they currently are.
2. **3 disguised duplicates** - copies of the *original clean text* of records that we
   damaged in steps 4 and 5b. Right now these do **not** match anything, because the
   record they came from is wrapped in HTML tags or padded with extra spaces. They only
   become visible duplicates *after* cleaning.

The second kind is the reason duplicate removal is the **last** step in the pipeline. We
will come back to this.

In [12]:
# --- 10 obvious duplicates: exact copies of rows as they are right now ---
duplicates = noisy_df.sample(10, random_state=42)

# --- 3 disguised duplicates: the ORIGINAL clean text of rows we damaged earlier ---
# Rows 0 and 1 currently have HTML tags; row 20 currently has messy whitespace.
disguised = pd.DataFrame({
    "clinical_text": [
        clean_df.loc[0, "clinical_text"],
        clean_df.loc[1, "clinical_text"],
        clean_df.loc[20, "clinical_text"],
    ]
})

noisy_df = pd.concat([noisy_df, duplicates, disguised], ignore_index=True)

duplicates_added = len(duplicates) + len(disguised)

print("Added", len(duplicates), "obvious duplicates and", len(disguised), "disguised duplicates.")
print("Records now:", len(noisy_df))
print()
print("A disguised duplicate and the record it copies:")
print("  damaged version:", repr(noisy_df.loc[0, "clinical_text"])[:85])
print("  clean copy     :", repr(disguised.loc[0, "clinical_text"])[:85])
print("  -> these are different strings today, and identical after cleaning.")

Added 10 obvious duplicates and 3 disguised duplicates.
Records now: 263

A disguised duplicate and the record it copies:
  damaged version: '<p> An example/template for meatotomy.</p>'
  clean copy     : ' An example/template for meatotomy.'
  -> these are different strings today, and identical after cleaning.


---
## STEP 7 - Add Empty Entries

Rows where the text is missing entirely. Three different kinds, because they behave
differently in pandas:

| Value | What it is |
|---|---|
| `""` | an empty string |
| `"   "` | only spaces - looks empty to a person, but is not an empty string to a computer |
| `None` | a missing value, which pandas shows as `NaN` |

In [13]:
empty_rows = pd.DataFrame({
    "clinical_text": ["", "   ", None, "", "   ", None]
})

noisy_df = pd.concat([noisy_df, empty_rows], ignore_index=True)

empty_added = len(empty_rows)

print("Added", empty_added, "empty records.")
print("Records now:", len(noisy_df))

Added 6 empty records.
Records now: 269


---
## STEP 8 - Add Very Short Notes

> **Very short records are removed because they usually contain insufficient information
> for instruction tuning.**

Entries like `"OK"` or `"N/A"` are real rows, but there is nothing in them for a model to
learn from. A training example built from `"OK"` would only teach a model to produce
vague, empty answers.

In [14]:
short_rows = pd.DataFrame({
    "clinical_text": ["OK", "Yes", "Pain", "Normal", "N/A", "Stable"]
})

noisy_df = pd.concat([noisy_df, short_rows], ignore_index=True)

short_added = len(short_rows)

print("Added", short_added, "very short records.")
print("Records now:", len(noisy_df))

Added 6 very short records.
Records now: 275


---
## STEP 9 - Add Excessively Long Records

A corrupted export can repeat the same text over and over. We create three such records by
taking a real record and repeating it.

Multiplying a string by a number repeats it: `"abc" * 3` gives `"abcabcabc"`.

In [15]:
# Build 3 over-long records by repeating real records from the dataset many times.
long_rows = pd.DataFrame({
    "clinical_text": [
        clean_df.loc[0, "clinical_text"] * 100,
        clean_df.loc[1, "clinical_text"] * 50,
        clean_df.loc[2, "clinical_text"] * 40,
    ]
})

noisy_df = pd.concat([noisy_df, long_rows], ignore_index=True)

long_added = len(long_rows)

print("Added", long_added, "excessively long records, with lengths:")
for text in long_rows["clinical_text"]:
    print(" ", len(text), "characters")
print()
print("Records now:", len(noisy_df))

Added 3 excessively long records, with lengths:
  3500 characters
  5300 characters
  3360 characters

Records now: 278


---
## STEP 10 - Show the Noisy Dataset

The dataset now contains all six problems:

1. **HTML tags** - `<p>`, `<div>`, `<span>` wrapped around the text
2. **OCR artifacts** - `Pat1ent`, `d1abetes`, `hypertens1on`
3. **Duplicate records** - 10 obvious and 3 disguised
4. **Empty records** - `""`, `"   "` and `None`
5. **Very short records** - `"OK"`, `"N/A"`, `"Stable"`
6. **Excessively long records** - real text repeated dozens of times

Plus messy whitespace scattered through 10 records.

In [16]:
noisy_df.head(20)

,clinical_text
0,<p> An example/template for meatotomy.</p>
1,"<div> Normal physical exam template. Normal appearance for chronological age, does no..."
2,<span> Neurologic consultation was requested to assess and assist with seizure medicat...
3,"<p> Multiple sharp force injuries, involving chest and abdomen, multiple incised-stab ..."
4,<div> The pat1ent with atypical type right arm discomfort and neck discomfort.</div>
5,<span> Implantation of a dual chamber permanent pacemaker</span>
6,<p> Right lower pole renal stone and possibly infected stent. Cysto stent removal.</p>
7,"<div> Incision and drainage of the penoscrotal abscess, packing, penile biopsy, cystos..."
8,<span> Stress test - Adenosine Myoview. Ischemic cardiomyopathy. Inferoseptal and ap...
9,"<p> Ruptured distal biceps tendon, right elbow. Repair of distal biceps tendon, right..."


In [17]:
print("Records after adding artificial noise:", len(noisy_df))
print()
print("Where those records came from:")
print(f"  Real records from MTSamples : {len(clean_df)}")
print(f"  + duplicates added          : {duplicates_added}")
print(f"  + empty records added       : {empty_added}")
print(f"  + very short records added  : {short_added}")
print(f"  + excessively long added    : {long_added}")
print(f"  {'-' * 38}")
print(f"  Total noisy records         : {len(noisy_df)}")
print()
print("Records damaged in place (the row count does not change for these):")
print(f"  HTML tags added        : {html_damaged_count}")
print(f"  OCR artifacts added    : {ocr_damaged_count}")
print(f"  Messy whitespace added : {whitespace_damaged_count}")

Records after adding artificial noise: 278

Where those records came from:
  Real records from MTSamples : 250
  + duplicates added          : 13
  + empty records added       : 6
  + very short records added  : 6
  + excessively long added    : 3
  --------------------------------------
  Total noisy records         : 278

Records damaged in place (the row count does not change for these):
  HTML tags added        : 15
  OCR artifacts added    : 20
  Messy whitespace added : 10


In [18]:
# Look at one example of each problem, so the damage is easy to see.
print("EXAMPLES OF EACH PROBLEM")
print("=" * 78)
print("\n1. HTML tags:")
print("  ", noisy_df.loc[0, "clinical_text"][:95])
print("\n2. OCR artifacts:")
print("  ", noisy_df.loc[ocr_target_rows[0], "clinical_text"][:95])
print("\n3. Messy whitespace:")
print("  ", repr(noisy_df.loc[20, "clinical_text"])[:95])
print("\n4. Empty records:")
print("  ", repr(noisy_df.loc[len(noisy_df) - 15, "clinical_text"]))
print("\n5. Very short records:")
print("  ", repr(noisy_df.loc[len(noisy_df) - 9, "clinical_text"]))
print("\n6. Excessively long records:")
print("  ", len(noisy_df.loc[len(noisy_df) - 3, "clinical_text"]), "characters, starting:",
      noisy_df.loc[len(noisy_df) - 3, "clinical_text"][:55], "...")

EXAMPLES OF EACH PROBLEM

1. HTML tags:
   <p> An example/template for meatotomy.</p>

2. OCR artifacts:
   <div> The pat1ent with atypical type right arm discomfort and neck discomfort.</div>

3. Messy whitespace:
   '       \nAn 84-year-old woman with a history of hypertens1on, severe tricuspid regurgitation w

4. Empty records:
   ''

5. Very short records:
   'OK'

6. Excessively long records:
   3500 characters, starting:  An example/template for meatotomy. An example/template ...


---
## STEP 11 - Dataset Statistics Before Cleaning

Before changing anything, we measure the dataset. Without a "before" measurement we cannot
show that the cleaning did anything.

We write it as a **function** so the exact same measurement can be repeated after cleaning
and the two compared fairly.

**What each statistic tells us:**

- **Total records** - how many rows there are.
- **Empty records** - rows with no text, or only spaces.
- **Duplicate records** - rows whose text already appeared in an earlier row.
- **Minimum / maximum / average text length** - in characters. A minimum of 0 means empty
  rows exist; a maximum in the thousands means at least one record is corrupted.

In [19]:
def dataset_statistics(df, title="DATASET STATISTICS"):
    """Print simple statistics about the clinical_text column of a DataFrame."""

    # fillna("") replaces missing values (None/NaN) with an empty string, so that
    # .str.len() and the comparisons below work on every row without errors.
    text = df["clinical_text"].fillna("").astype(str)

    total_records = len(df)
    empty_records = (text.str.strip() == "").sum()   # .strip() removes spaces at both ends
    duplicate_records = text.duplicated().sum()      # True if seen in an earlier row
    lengths = text.str.len()                         # number of characters in each row

    print(f"===== {title} =====")
    print()
    print(f"Total records:       {total_records}")
    print(f"Empty records:       {empty_records}")
    print(f"Duplicate records:   {duplicate_records}")
    print(f"Minimum text length: {lengths.min()}")
    print(f"Maximum text length: {lengths.max()}")
    print(f"Average text length: {round(lengths.mean())}")
    print()


# Keep a snapshot of the dataset exactly as it is now, before any cleaning happens.
# We need it later to print the before-and-after comparison and to show genuine
# "before" examples, rather than trying to reconstruct them from memory.
noisy_df_before_cleaning = noisy_df.copy()
records_before = len(noisy_df_before_cleaning)

dataset_statistics(noisy_df, "BEFORE CLEANING")

===== BEFORE CLEANING =====

Total records:       278
Empty records:       6
Duplicate records:   18
Minimum text length: 0
Maximum text length: 5300
Average text length: 173



Two of these numbers deserve a second look.

**Duplicate records.** This counts only rows that are *character-for-character* identical
to an earlier row. Our three disguised duplicates are **not** counted yet, because right
now they are not identical to anything - one is wrapped in HTML tags, another is padded
with spaces. They will only be caught once cleaning has removed those. **That is exactly
why duplicate removal is the last step.**

Also note that `fillna("")` turns the `None` rows into `""`, so they match the empty
strings we added and count as duplicates too. They are removed as empty records anyway.

**Minimum text length 0** proves empty rows exist, and a **maximum in the thousands**
proves at least one record is corrupted.

---
## STEP 12 - Clean HTML Tags

### What is a regular expression?

A regular expression (regex) describes a *pattern* of text rather than exact words. We use
`<.*?>`, which reads as:

| piece | meaning |
|---|---|
| `<` | a literal `<` character |
| `.` | any single character |
| `*` | the previous thing, repeated any number of times |
| `?` | stop at the **first** `>`, not the last one |
| `>` | a literal `>` character |

So `<.*?>` means *"a `<`, then anything, then the first `>`"* - exactly what an HTML tag
looks like. `re.sub(pattern, "", text)` finds every match and replaces it with an empty
string, deleting it.

The `?` matters. Without it, `<p>Fever</p>` would match from the first `<` all the way to
the last `>` and delete the word "Fever" too. With `?`, the two tags are removed
separately and the text between them survives.

> **This removes HTML tags while keeping the text inside them.**

**We do this first** because it changes what the text says, and every later step should
work on the real words.

In [20]:
def remove_html(text):
    """Remove HTML tags such as <p>, </div> or <span> from a piece of text."""
    return re.sub(r"<.*?>", "", text)


# Try it on one example first, to check it behaves the way we expect.
example = "<p>Patient reports fever and cough.</p>"
print("Before:", example)
print("After: ", remove_html(example))

Before: <p>Patient reports fever and cough.</p>
After:  Patient reports fever and cough.


In [21]:
# Replace missing values with empty strings FIRST. Our cleaning functions expect text,
# and calling .replace() on None would cause an error.
noisy_df["clinical_text"] = noisy_df["clinical_text"].fillna("").astype(str)

# .apply() runs our function on every row of the column, one row at a time.
noisy_df["clinical_text"] = noisy_df["clinical_text"].apply(remove_html)

print("HTML tags removed. The records that had them:\n")
for i in range(3):
    print(" ", noisy_df.loc[i, "clinical_text"][:95])

remaining_tags = noisy_df["clinical_text"].str.contains("<.*?>", regex=True).sum()
print("\nRecords still containing an HTML tag:", remaining_tags)

HTML tags removed. The records that had them:

   An example/template for meatotomy.
   Normal physical exam template.  Normal appearance for chronological age, does not appear chron
   Neurologic consultation was requested to assess and assist with seizure medication.

Records still containing an HTML tag: 0


---
## STEP 13 - Fix OCR Artifacts

We fix the scanning mistakes with a **dictionary** of known wrong spellings and their
corrections. A Python dictionary stores pairs - here *wrong text -> correct text* - and we
loop through them calling `.replace()` for each.

This dictionary is the exact reverse of the one used in STEP 5.

> **Be honest about what this is.** It is a *demonstration*, not a real OCR corrector. It
> only fixes mistakes that are already on the list. Real OCR correction uses spell-checking
> against a medical dictionary or a language model, which is far beyond this task. The
> point is to show the *shape* of the fix: find known errors, replace them.

In [22]:
# The corrections: what the scanner produced -> what it should be.
ocr_corrections = {
    "Pat1ent": "Patient",
    "pat1ent": "patient",
    "d1abetes": "diabetes",
    "hypertens1on": "hypertension",
}


def fix_ocr(text):
    """Replace known OCR mistakes using the dictionary above."""
    for wrong, correct in ocr_corrections.items():
        text = text.replace(wrong, correct)
    return text


# Check it on one example before applying it to the whole dataset.
example = "Pat1ent with d1abetes and hypertens1on."
print("Before:", example)
print("After: ", fix_ocr(example))

Before: Pat1ent with d1abetes and hypertens1on.
After:  Patient with diabetes and hypertension.


In [23]:
noisy_df["clinical_text"] = noisy_df["clinical_text"].apply(fix_ocr)

print("OCR errors fixed. The records that had them:\n")
for i in ocr_target_rows[:3]:
    print(" ", noisy_df.loc[i, "clinical_text"][:95])

# Check that none of the damaged spellings survive anywhere in the dataset.
remaining_ocr = noisy_df["clinical_text"].str.contains(
    "|".join(ocr_corrections.keys()), regex=True
).sum()
print("\nRecords still containing an OCR error:", remaining_ocr)

OCR errors fixed. The records that had them:

   The patient with atypical type right arm discomfort and neck discomfort.
   The patient is a 60-year-old female patient who off and on for the past 10 to 12 months has ha
   The patient is a 58-year-old African-American right-handed female with 16 years of education w

Records still containing an OCR error: 0


---
## STEP 14 - Normalize Whitespace

"Whitespace" means spaces, tab characters and line breaks. Messy text has too many of them.

The pattern `\s+` means *"one or more whitespace characters"*. Replacing every match with
a single space turns any run of spaces or line breaks into one normal space. `.strip()`
then removes any space left at the beginning or end.

**Before:**

```text
   Patient reports
fever and cough.
```

**After:**

```text
Patient reports fever and cough.
```

This matters more than it looks. `"fever and cough"` and `"fever  and cough"` are
*different strings* to a computer, so a duplicate check would not match them. Normalizing
whitespace first is what allows STEP 18 to catch those duplicates.

In [24]:
def normalize_whitespace(text):
    """Turn any run of spaces, tabs or line breaks into a single space."""
    return re.sub(r"\s+", " ", text).strip()


# Check it on one example.
example = "   Patient reports    \n fever and cough.   "
print("Before:", repr(example))
print("After: ", repr(normalize_whitespace(example)))

Before: '   Patient reports    \n fever and cough.   '
After:  'Patient reports fever and cough.'


In [25]:
noisy_df["clinical_text"] = noisy_df["clinical_text"].apply(normalize_whitespace)

print("Whitespace normalized. The messy records:\n")
for i in whitespace_target_rows[:3]:
    print(" ", repr(noisy_df.loc[i, "clinical_text"])[:95])

Whitespace normalized. The messy records:

  'An 84-year-old woman with a history of hypertension, severe tricuspid regurgitation with mild 
  'Left-sided large hemicraniectomy for traumatic brain injury and increased intracranial pressur
  'Left partial nephrectomy due to left renal mass.'


---
## STEP 15 - Remove Empty Entries

The text cleaning is finished. From here on we delete whole *rows*.

> **Empty clinical notes do not contain useful information, so they are removed.**

Note that after normalizing whitespace, the rows that contained only spaces are now
genuinely empty strings - another example of an earlier step setting up a later one.

The line below reads as: *keep only the rows where the text, with outside spaces removed,
is not empty.*

In [26]:
records_before_step = len(noisy_df)

noisy_df = noisy_df[noisy_df["clinical_text"].str.strip() != ""]

print("Removed", records_before_step - len(noisy_df), "empty records.")
print("Records remaining:", len(noisy_df))

Removed 6 empty records.
Records remaining: 272


---
## STEP 16 - Remove Very Short Notes

> **Notes shorter than 20 characters are considered too short to provide useful clinical
> information for this demonstration.**

**Why 20?** It is a judgement call, and it should be checked rather than guessed. The junk
entries we added (`"OK"`, `"N/A"`) are all under 7 characters, while genuine MTSamples
descriptions are typically 40 to 400. Any threshold in that gap would work.

**Watch what gets removed.** Some of the records deleted here are *genuine* dataset
records that happen to be very short, not noise we added. That is a real and honest
consequence of a length filter: it cannot tell a useless real record from a useless fake
one. The cell prints exactly what it deletes so this is visible rather than hidden.

In [27]:
MIN_LENGTH = 20   # minimum number of characters for a note to be kept

records_before_step = len(noisy_df)

# Show what we are about to delete - always worth checking before removing data.
too_short = noisy_df[noisy_df["clinical_text"].str.len() < MIN_LENGTH]
print("Records shorter than", MIN_LENGTH, "characters:")
for text in too_short["clinical_text"]:
    print(f"  {len(text):3d} chars  {repr(text)}")

noisy_df = noisy_df[noisy_df["clinical_text"].str.len() >= MIN_LENGTH]

print()
print("Removed", records_before_step - len(noisy_df), "very short records.")
print("Records remaining:", len(noisy_df))

Records shorter than 20 characters:
   19 chars  'PICC line insertion'
    2 chars  'OK'
    3 chars  'Yes'
    4 chars  'Pain'
    6 chars  'Normal'
    3 chars  'N/A'
    6 chars  'Stable'

Removed 7 very short records.
Records remaining: 265


---
## STEP 17 - Remove Excessively Long Records

> **Records longer than 1000 characters are removed as excessively long records for this
> simple preprocessing pipeline.**

**Why remove them?** Two reasons:

1. **They are usually broken.** A record thousands of characters long that repeats the
   same sentence is a corrupted export, not a real clinical note. Training on it teaches
   the model to repeat itself.
2. **They are expensive.** Long examples use far more memory during training, and one huge
   record can slow down an entire batch.

**Why 1000?** The longest genuine `description` in the full MTSamples file is 492
characters, so 1000 leaves a generous margin while still catching the corrupted records we
built. A dataset of full discharge summaries would need a far larger limit - this number
is specific to this data.

In [28]:
MAX_LENGTH = 1000   # maximum number of characters for a note to be kept

records_before_step = len(noisy_df)

# Show what we are about to delete, shortened so it does not flood the screen.
too_long = noisy_df[noisy_df["clinical_text"].str.len() > MAX_LENGTH]
print("Records longer than", MAX_LENGTH, "characters:")
for text in too_long["clinical_text"]:
    print(f"  {len(text):6d} chars, starting with: {text[:55]}...")

noisy_df = noisy_df[noisy_df["clinical_text"].str.len() <= MAX_LENGTH]

print()
print("Removed", records_before_step - len(noisy_df), "excessively long records.")
print("Records remaining:", len(noisy_df))

Records longer than 1000 characters:
    3499 chars, starting with: An example/template for meatotomy. An example/template ...
    5249 chars, starting with: Normal physical exam template. Normal appearance for ch...
    3359 chars, starting with: Neurologic consultation was requested to assess and ass...

Removed 3 excessively long records.
Records remaining: 262


---
## STEP 18 - Remove Duplicate Records

> **If the same clinical note appears more than once, keep only one copy.**

**Why?** A note appearing twice would be shown to the model twice during training, which
makes the model treat that example as more important than the others for no good reason.

`drop_duplicates(subset=["clinical_text"])` keeps the **first** appearance of each note and
deletes any later copies. `subset=["clinical_text"]` tells pandas to compare *only* the
text column.

**This is the last cleaning step, on purpose.** Duplicate detection compares text exactly,
so it only works once every version of a note looks the same. Our three disguised
duplicates were invisible at the start - hidden behind HTML tags and extra spaces. Now that
STEP 12 and STEP 14 have removed those, all copies match and can be caught.

Some of the duplicates removed here are also **pairs that were already in the real
MTSamples data**, not ones we added.

In [29]:
records_before_step = len(noisy_df)

# Count how many notes appear more than once, before deleting anything.
duplicate_mask = noisy_df["clinical_text"].duplicated(keep=False)   # keep=False marks ALL copies
duplicated_texts = sorted(set(noisy_df.loc[duplicate_mask, "clinical_text"]))

print("Distinct notes that appear more than once:", len(duplicated_texts))
print("\nFirst 5 of them:\n")
for text in duplicated_texts[:5]:
    count = (noisy_df["clinical_text"] == text).sum()
    print(f"  {count} copies: {text[:78]}")

noisy_df = noisy_df.drop_duplicates(subset=["clinical_text"])

print()
print("Removed", records_before_step - len(noisy_df), "duplicate records.")
print("Records remaining:", len(noisy_df))

Distinct notes that appear more than once: 16

First 5 of them:

  3 copies: An 84-year-old woman with a history of hypertension, severe tricuspid regurgit
  2 copies: An example/template for meatotomy.
  2 copies: Arthroscopic subacromial decompression and repair of rotator cuff through mini
  2 copies: Burr hole and insertion of external ventricular drain catheter.
  2 copies: Complete laminectomy, L4. and facetectomy, L3-L4 level. A dural repair, right 

Removed 17 duplicate records.
Records remaining: 245


---
## STEP 19 - Reset the Index

Filtering rows leaves gaps in the row numbers - after deleting row 3, the index jumps
2, 4, 5. `reset_index(drop=True)` renumbers them 0, 1, 2, ... and `drop=True` throws the
old numbering away instead of keeping it as a column.

In [30]:
noisy_df = noisy_df.reset_index(drop=True)

print("Index reset. The cleaned dataset now has", len(noisy_df), "records,")
print("numbered 0 to", len(noisy_df) - 1, "with no gaps.")

Index reset. The cleaned dataset now has 245 records,
numbered 0 to 244 with no gaps.


---
## STEP 20 - Statistics After Cleaning

Now we run the **same function** from STEP 11 on the cleaned data and compare.

In [31]:
# Print both sets of statistics together, using the snapshot we saved in STEP 11.
dataset_statistics(noisy_df_before_cleaning, "BEFORE CLEANING")
dataset_statistics(noisy_df, "AFTER CLEANING")

records_after = len(noisy_df)
records_removed = records_before - records_after

print("Records removed:  ", records_removed)
print("Records remaining:", records_after)
print(f"Percentage kept:   {records_after / records_before * 100:.1f}%")

===== BEFORE CLEANING =====

Total records:       278
Empty records:       6
Duplicate records:   18
Minimum text length: 0
Maximum text length: 5300
Average text length: 173

===== AFTER CLEANING =====

Total records:       245
Empty records:       0
Duplicate records:   0
Minimum text length: 23
Maximum text length: 389
Average text length: 133

Records removed:   33
Records remaining: 245
Percentage kept:   88.1%


In [32]:
# A breakdown of where the removed records went, so the total is accounted for.
print("WHAT WAS REMOVED, AND WHY")
print("=" * 78)
print()
print("Records we deliberately added as noise:")
print(f"  duplicates added        : {duplicates_added}")
print(f"  empty records added     : {empty_added}")
print(f"  very short records added: {short_added}")
print(f"  excessively long added  : {long_added}")
print(f"  {'-' * 40}")
print(f"  total noise records     : {duplicates_added + empty_added + short_added + long_added}")
print()
print("Plus problems that were ALREADY in the real MTSamples data:")
print(f"  real records under 20 characters : {(existing_lengths < 20).sum()}")
print(f"  real duplicate records           : {existing_text.duplicated().sum()}")
print()
print(f"Total removed: {records_removed}")
print()
print("This is worth understanding: not every deleted record was noise we injected.")
print("A real dataset arrives with its own duplicates and useless short entries, and")
print("the same filters remove those too. That is the pipeline working correctly.")

WHAT WAS REMOVED, AND WHY

Records we deliberately added as noise:
  duplicates added        : 13
  empty records added     : 6
  very short records added: 6
  excessively long added  : 3
  ----------------------------------------
  total noise records     : 28

Plus problems that were ALREADY in the real MTSamples data:
  real records under 20 characters : 1
  real duplicate records           : 4

Total removed: 33

This is worth understanding: not every deleted record was noise we injected.
A real dataset arrives with its own duplicates and useless short entries, and
the same filters remove those too. That is the pipeline working correctly.


Every number moved in the right direction:

- **Empty records** went to 0 - no blank rows are left.
- **Duplicate records** went to 0 - every note is now unique.
- **Minimum length** rose from 0 to a real sentence length.
- **Maximum length** dropped from thousands of characters back to a normal note.

In [33]:
# An automatic check that the cleaning actually achieved what it claims.
final_text = noisy_df["clinical_text"]

checks = {
    "no empty records":        (final_text.str.strip() == "").sum() == 0,
    "no duplicate records":    final_text.duplicated().sum() == 0,
    "no HTML tags left":       final_text.str.contains("<.*?>", regex=True).sum() == 0,
    "no OCR errors left":      final_text.str.contains("|".join(ocr_corrections.keys()), regex=True).sum() == 0,
    "all records >= MIN_LENGTH": final_text.str.len().min() >= MIN_LENGTH,
    "all records <= MAX_LENGTH": final_text.str.len().max() <= MAX_LENGTH,
    "no leading/trailing spaces": (final_text != final_text.str.strip()).sum() == 0,
}

print("FINAL QUALITY CHECKS")
print("=" * 78)
for description, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {description}")

print()
print("All checks passed." if all(checks.values()) else "SOME CHECKS FAILED - review above.")

FINAL QUALITY CHECKS
  PASS  no empty records
  PASS  no duplicate records
  PASS  no HTML tags left
  PASS  no OCR errors left
  PASS  all records >= MIN_LENGTH
  PASS  all records <= MAX_LENGTH
  PASS  no leading/trailing spaces

All checks passed.


---
## STEP 21 - Display Cleaned Dataset

The final check is to read the data. Every row should be readable clinical text with no
tags, no digits inside words, and no strange spacing.

In [34]:
noisy_df.head(10)

,clinical_text
0,An example/template for meatotomy.
1,"Normal physical exam template. Normal appearance for chronological age, does not appea..."
2,Neurologic consultation was requested to assess and assist with seizure medication.
3,"Multiple sharp force injuries, involving chest and abdomen, multiple incised-stab woun..."
4,The patient with atypical type right arm discomfort and neck discomfort.
5,Implantation of a dual chamber permanent pacemaker
6,Right lower pole renal stone and possibly infected stent. Cysto stent removal.
7,"Incision and drainage of the penoscrotal abscess, packing, penile biopsy, cystoscopy, ..."
8,Stress test - Adenosine Myoview. Ischemic cardiomyopathy. Inferoseptal and apical tran...
9,"Ruptured distal biceps tendon, right elbow. Repair of distal biceps tendon, right elbow."


In [35]:
# Compare each damaged record with its cleaned version. The "before" text is the real
# damaged record, taken from the snapshot saved in STEP 11.
#
# Note: we cannot simply read row i of noisy_df, because rows were deleted and the index
# was reset in STEP 19, so row i is no longer the same record. Instead we run the same
# three cleaning functions on the "before" text, which is exactly what the pipeline did.
def show_before_after(label, row_numbers):
    """Print the before and after text for the given rows."""
    print(f"--- {label} ---")
    for i in row_numbers:
        before = noisy_df_before_cleaning.loc[i, "clinical_text"]
        after = normalize_whitespace(fix_ocr(remove_html(before)))
        print(f"  before: {repr(before)[:88]}")
        print(f"  after : {repr(after)[:88]}")
        print()


print("BEFORE AND AFTER, RECORD BY RECORD")
print("=" * 78)
print()
show_before_after("Records that had HTML tags", [0, 1, 2])
show_before_after("Records that had OCR artifacts", ocr_target_rows[:2])
show_before_after("Records that had messy whitespace", whitespace_target_rows[:2])

BEFORE AND AFTER, RECORD BY RECORD

--- Records that had HTML tags ---
  before: '<p> An example/template for meatotomy.</p>'
  after : 'An example/template for meatotomy.'

  before: '<div> Normal physical exam template.  Normal appearance for chronological age, does not
  after : 'Normal physical exam template. Normal appearance for chronological age, does not appear

  before: '<span> Neurologic consultation was requested to assess and assist with seizure medicati
  after : 'Neurologic consultation was requested to assess and assist with seizure medication.'

--- Records that had OCR artifacts ---
  before: '<div> The pat1ent with atypical type right arm discomfort and neck discomfort.</div>'
  after : 'The patient with atypical type right arm discomfort and neck discomfort.'

  before: '<div> The pat1ent is a 60-year-old female pat1ent who off and on for the past 10 to 12 
  after : 'The patient is a 60-year-old female patient who off and on for the past 10 to 12 months

--- Record

In [36]:
# Ten complete cleaned records, printed in full.
print("TEN CLEANED CLINICAL RECORDS")
print("=" * 78)
for i in range(10):
    print(f"\n{i}: {noisy_df.loc[i, 'clinical_text']}")

TEN CLEANED CLINICAL RECORDS

0: An example/template for meatotomy.

1: Normal physical exam template. Normal appearance for chronological age, does not appear chronically ill.

2: Neurologic consultation was requested to assess and assist with seizure medication.

3: Multiple sharp force injuries, involving chest and abdomen, multiple incised-stab wounds of the neck, and multiple incised or cutting wounds.

4: The patient with atypical type right arm discomfort and neck discomfort.

5: Implantation of a dual chamber permanent pacemaker

6: Right lower pole renal stone and possibly infected stent. Cysto stent removal.

7: Incision and drainage of the penoscrotal abscess, packing, penile biopsy, cystoscopy, and urethral dilation.

8: Stress test - Adenosine Myoview. Ischemic cardiomyopathy. Inferoseptal and apical transmural scar.

9: Ruptured distal biceps tendon, right elbow. Repair of distal biceps tendon, right elbow.


---
## STEP 22 - Convert to Instruction-Tuning Format

### What is instruction tuning?

Instruction tuning teaches a model to *follow instructions* by showing it many examples of
a request paired with a good response. Each training example is a short conversation:

- a **user** message - the instruction,
- an **assistant** message - the response we want the model to learn to produce.

```json
{
  "messages": [
    {"role": "user",      "content": "Analyze this clinical note: ..."},
    {"role": "assistant", "content": "The clinical note describes ..."}
  ]
}
```

### How we build the response

The user message is a fixed instruction with the note appended. The assistant message is a
simple, safe response. To make the responses slightly less repetitive we check whether a
few common clinical words appear in the note - a plain keyword search, no machine learning.

> ### An important safety rule
>
> The assistant replies **never contain a diagnosis or medical advice**. They only note
> what kind of information the record contains and direct the reader to a qualified
> professional. Writing invented diagnoses into a training file would teach a future model
> to produce invented diagnoses - exactly the harm to avoid in medical data.
>
> **This project is about data preprocessing, not medical diagnosis.**

In [37]:
# A few broad categories, matched by simple keyword search.
CATEGORY_KEYWORDS = {
    "a surgical procedure":        ["surgery", "surgical", "excision", "repair", "incision",
                                    "laparoscopic", "resection", "arthroscopic"],
    "a diagnostic examination":    ["x-ray", "mri", "ct ", "ultrasound", "echocardiogram",
                                    "biopsy", "endoscopy", "scan"],
    "a patient consultation":      ["consult", "history", "presents", "complaint",
                                    "follow-up", "evaluation"],
}


def describe_record(text):
    """Return a short, safe description of what kind of record this is."""
    lowered = text.lower()
    for description, keywords in CATEGORY_KEYWORDS.items():
        for keyword in keywords:
            if keyword in lowered:
                return description
    return "clinical information"


def build_instruction_record(text):
    """Turn one cleaned clinical note into one instruction-tuning example."""
    category = describe_record(text)
    assistant_reply = (
        f"The clinical note describes {category}. "
        "This is a summary of the recorded text only and not a diagnosis. "
        "It should be reviewed by a qualified healthcare professional."
    )
    return {
        "messages": [
            {"role": "user", "content": f"Analyze this clinical note: {text}"},
            {"role": "assistant", "content": assistant_reply},
        ]
    }


# Build one training example for every cleaned record.
instruction_records = [build_instruction_record(t) for t in noisy_df["clinical_text"]]

print("Created", len(instruction_records), "instruction-tuning examples.")
print()
print("The first one, printed neatly:")
print(json.dumps(instruction_records[0], indent=2))

Created 245 instruction-tuning examples.

The first one, printed neatly:
{
  "messages": [
    {
      "role": "user",
      "content": "Analyze this clinical note: An example/template for meatotomy."
    },
    {
      "role": "assistant",
      "content": "The clinical note describes clinical information. This is a summary of the recorded text only and not a diagnosis. It should be reviewed by a qualified healthcare professional."
    }
  ]
}


---
## STEP 23 - Export JSONL

**JSONL** stands for **JSON Lines**: a text file where *each line is one complete JSON
object*. It is the usual format for training data because a program can read one example
at a time without loading the whole file into memory.

`json.dumps(record)` turns a Python dictionary into a JSON string, and we add `"\n"` so
the next record starts on a new line. `ensure_ascii=False` keeps accented characters
readable instead of turning them into escape codes.

In [38]:
jsonl_filename = "clinical_instruction_dataset.jsonl"

# "w" means write; encoding="utf-8" makes sure all characters are saved correctly.
with open(jsonl_filename, "w", encoding="utf-8") as f:
    for record in instruction_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved", len(instruction_records), "records to", jsonl_filename)
print("File size:", round(os.path.getsize(jsonl_filename) / 1024, 1), "KB")

Saved 245 records to clinical_instruction_dataset.jsonl
File size: 101.4 KB


---
## STEP 24 - Export Cleaned CSV

We also save the cleaned table as a CSV. This is the human-readable deliverable - it opens
in Excel and can be checked by hand.

`index=False` stops pandas writing its internal row numbers as an extra column.

In [39]:
csv_filename = "cleaned_clinical_dataset.csv"

noisy_df.to_csv(csv_filename, index=False, encoding="utf-8")

print("Saved", len(noisy_df), "records to", csv_filename)
print("File size:", round(os.path.getsize(csv_filename) / 1024, 1), "KB")

Saved 245 records to cleaned_clinical_dataset.csv
File size: 32.3 KB


---
## STEP 25 - Display Sample JSONL Records

Writing a file is not proof that the file is correct. We read it back and check.

In [40]:
# Read the first 3 lines exactly as they were written.
print("THE FIRST 3 LINES OF THE FILE")
print("=" * 78)
with open(jsonl_filename, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f.readline())

THE FIRST 3 LINES OF THE FILE
{"messages": [{"role": "user", "content": "Analyze this clinical note: An example/template for meatotomy."}, {"role": "assistant", "content": "The clinical note describes clinical information. This is a summary of the recorded text only and not a diagnosis. It should be reviewed by a qualified healthcare professional."}]}

{"messages": [{"role": "user", "content": "Analyze this clinical note: Normal physical exam template. Normal appearance for chronological age, does not appear chronically ill."}, {"role": "assistant", "content": "The clinical note describes clinical information. This is a summary of the recorded text only and not a diagnosis. It should be reviewed by a qualified healthcare professional."}]}

{"messages": [{"role": "user", "content": "Analyze this clinical note: Neurologic consultation was requested to assess and assist with seizure medication."}, {"role": "assistant", "content": "The clinical note describes a patient consultation. This i

In [41]:
# Load every line back as JSON, which also proves the whole file is valid.
with open(jsonl_filename, "r", encoding="utf-8") as f:
    loaded_records = [json.loads(line) for line in f]

print("Records read back from the file:", len(loaded_records))
print("=" * 78)

for i, record in enumerate(loaded_records[:3], start=1):
    print(f"\nRECORD {i}")
    for message in record["messages"]:
        print(f"  {message['role'].upper()}:")
        print(f"    {message['content']}")

print()
print("=" * 78)
print("Every line loaded successfully, so the JSONL file is valid.")

Records read back from the file: 245

RECORD 1
  USER:
    Analyze this clinical note: An example/template for meatotomy.
  ASSISTANT:
    The clinical note describes clinical information. This is a summary of the recorded text only and not a diagnosis. It should be reviewed by a qualified healthcare professional.

RECORD 2
  USER:
    Analyze this clinical note: Normal physical exam template. Normal appearance for chronological age, does not appear chronically ill.
  ASSISTANT:
    The clinical note describes clinical information. This is a summary of the recorded text only and not a diagnosis. It should be reviewed by a qualified healthcare professional.

RECORD 3
  USER:
    Analyze this clinical note: Neurologic consultation was requested to assess and assist with seizure medication.
  ASSISTANT:
    The clinical note describes a patient consultation. This is a summary of the recorded text only and not a diagnosis. It should be reviewed by a qualified healthcare professional.

Ever

---
# STEP 26 - Preprocessing Decisions

| Problem | Solution |
|---|---|
| HTML tags | Removed using regular expressions |
| OCR artifacts | Corrected using a small replacement dictionary |
| Duplicate records | Removed using `drop_duplicates()` |
| Empty records | Removed |
| Very short notes | Removed using minimum length = 20 |
| Excessively long records | Removed using maximum length = 1000 |
| Extra whitespace | Normalized using regular expressions |
| Instruction format | Converted to `messages` format |
| Output format | JSONL |
| Clean dataset | CSV |

### Why the order matters

The steps are not interchangeable. Two rules decide the order:

1. **Fix the text before filtering the rows.** HTML removal, OCR correction and whitespace
   normalization all change what a record *says*. Filtering first would judge rows on text
   we were about to rewrite - a row of only spaces does not look empty until after
   normalization.

2. **Remove duplicates last.** Duplicate detection compares text character by character, so
   it only finds copies once every version of a note looks the same. Three of our
   duplicates were invisible at the start - hidden behind HTML tags and extra spaces.
   Running `drop_duplicates()` first would have missed all three.

### Choices worth defending in a viva

- **The `description` column, not `transcription`.** Full transcriptions have a median
  length of 2,667 characters, so a 1,000-character limit would have deleted the entire
  dataset. The choice of column and the choice of threshold have to agree with each other.
- **20 and 1000 characters are judgement calls, not standards.** They were chosen by
  looking at the real length distribution and are specific to this dataset.
- **The OCR dictionary only fixes the four errors we listed.** It demonstrates the
  approach; it is not a general OCR corrector.
- **Exact duplicate matching only.** Two notes that differ by a single word are kept as
  separate records. Fuzzy matching would be more thorough but much harder to explain, and
  the assignment only calls for exact duplicates.
- **The assistant replies never diagnose.** They describe the type of record and refer the
  reader to a professional.

---
# STEP 27 - Final Conclusion

In [42]:
print("=" * 78)
print("CLINICAL DATASET PREPARATION - SUMMARY")
print("=" * 78)
print()
print(f"  Source dataset:          MTSamples ({len(clean_df)} records selected)")
print(f"  Records before cleaning: {records_before}")
print(f"  Records removed:         {records_removed}")
print(f"  Records after cleaning:  {records_after}")
print(f"  Percentage kept:         {records_after / records_before * 100:.1f}%")
print()
print("  Files created:")
print(f"    {csv_filename}          ({len(noisy_df)} rows)")
print(f"    {jsonl_filename}  ({len(instruction_records)} training examples)")
print()
print("=" * 78)

CLINICAL DATASET PREPARATION - SUMMARY

  Source dataset:          MTSamples (250 records selected)
  Records before cleaning: 278
  Records removed:         33
  Records after cleaning:  245
  Percentage kept:         88.1%

  Files created:
    cleaned_clinical_dataset.csv          (245 rows)
    clinical_instruction_dataset.jsonl  (245 training examples)



### What this pipeline did

The pipeline starts with **MTSamples**, a publicly available clinical dataset, and selects
**250 records** of real medical text.

Artificial noise is then introduced to simulate the problems found in real-world clinical
data: **HTML tags** from web exports, **OCR artifacts** from scanned documents, **duplicate
records** from merged files, **empty records**, **uninformative short records**,
**excessively long corrupted records**, and **messy whitespace**.

The pipeline then removes or corrects every one of them, in an order chosen so that each
step sets up the next: text is repaired first (HTML, OCR, whitespace), rows are filtered
second (empty, short, long), and duplicates are removed last, once every copy of a note
finally looks identical.

The cleaned dataset is converted into JSONL instruction-tuning format, with safe assistant
responses that contain no diagnoses.

### Why the final dataset is more suitable for instruction tuning

- **Every record carries real information.** Empty rows and one-word entries are gone, so
  the model is never shown an example with nothing to learn from.
- **Every record is unique.** No note is over-weighted by appearing more than once.
- **The text is clean.** No HTML tags to reproduce, no `Pat1ent` misspellings to learn as
  real words, no random spacing.
- **Lengths are consistent.** No corrupted record dominates a training batch or teaches the
  model to repeat itself.
- **The format is ready to use.** Each record is a `messages` conversation, which is what
  fine-tuning tools expect.
- **The responses are safe.** No invented diagnosis appears anywhere in the training file.

### Limitations, stated honestly

- **OCR correction uses a small manually defined dictionary.** Only the four listed errors
  are fixed; real scanned text produces many others that would pass through untouched.
- **Duplicate removal only detects exact duplicates.** Two notes differing by one word are
  both kept.
- **The length thresholds are simple demonstration thresholds**, chosen for this dataset,
  and would need re-checking on any other data.
- **The assistant responses are generated from a keyword list**, so they are formulaic.
  Real training responses would be written or reviewed by people with clinical knowledge.
- **~250 records is far too few to actually fine-tune a model.** Real instruction tuning
  needs thousands. This demonstrates the *process*, not a production dataset.
- **This dataset is prepared for demonstration and future instruction tuning. It is not a
  clinical diagnostic system** and must not be used to make medical decisions.